# MelodyMatchMaker

This notebook develops a hybrid music recommendation system using Spotify data. It combines content-based filtering with popularity weighting for a hybrid approach, optimized for recommendations under 0.5 seconds using Annoy for approximate nearest neighbors.

In [ ]:
#install the required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree
import time
import ipywidgets as widgets
from IPython.display import display, HTML

In [2]:
%pip install pandas numpy scikit-learn ipywidgets

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl (12.4 MB)
Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl (8.1 MB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Load datasets
high_pop = pd.read_csv('high_popularity_spotify_data.csv')
low_pop = pd.read_csv('low_popularity_spotify_data.csv')

# Combine datasets
data = pd.concat([high_pop, low_pop], ignore_index=True)

# Extract year from release date
data['year'] = pd.to_datetime(data['track_album_release_date'], errors='coerce').dt.year

# Function to format duration
def format_duration(ms):
    seconds = ms / 1000
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes}:{secs:02d}"

# Display basic info
print(f"Total tracks: {len(data)}")
print(data.head())

Total tracks: 4831
   energy    tempo  danceability playlist_genre  loudness  liveness  valence  \
0   0.592  157.969         0.521            pop    -7.777     0.122    0.535   
1   0.507  104.978         0.747            pop   -10.171     0.117    0.438   
2   0.808  108.548         0.554            pop    -4.169     0.159    0.372   
3   0.910  112.966         0.670            pop    -4.070     0.304    0.786   
4   0.783  149.027         0.777            pop    -4.477     0.355    0.939   

            track_artist  time_signature  speechiness  ...  \
0  Lady Gaga, Bruno Mars             3.0       0.0304  ...   
1          Billie Eilish             4.0       0.0358  ...   
2          Gracie Abrams             4.0       0.0368  ...   
3      Sabrina Carpenter             4.0       0.0634  ...   
4       ROSÉ, Bruno Mars             4.0       0.2600  ...   

           track_album_id mode  key duration_ms acousticness  \
0  10FLjwfpbxLmW8c25Xyc2N  0.0  6.0    251668.0       0.3080   

In [15]:
# Select audio features for similarity
features = ['energy', 'tempo', 'danceability', 'loudness', 'liveness', 'valence', 'speechiness', 'instrumentalness', 'acousticness']

# Normalize features
scaler = StandardScaler()
data_features = scaler.fit_transform(data[features])

# For hybrid: include popularity as weight
data['popularity_weight'] = data['track_popularity'] / 100.0  # normalize to 0-1

print("Features shape:", data_features.shape)

Features shape: (4831, 9)


In [16]:
# Handle missing values
data[features] = data[features].fillna(data[features].mean())

# Normalize features
scaler = StandardScaler()
data_features = scaler.fit_transform(data[features])

# For hybrid: include popularity as weight
data['popularity_weight'] = data['track_popularity'] / 100.0  # normalize to 0-1

print("Features shape:", data_features.shape)
print("Any NaN in features:", np.isnan(data_features).any())

Features shape: (4831, 9)
Any NaN in features: False


In [9]:
# Build BallTree index for fast NN search
tree = BallTree(data_features, leaf_size=40)  # leaf_size for balance
print("BallTree index built")

BallTree index built


In [10]:
def get_recommendations(track_index, n=5):
    start_time = time.time()
    
    # Get nearest neighbors using BallTree
    query_point = data_features[track_index].reshape(1, -1)
    distances, indices = tree.query(query_point, k=n+1)  # +1 to exclude self
    
    # Flatten
    distances = distances[0][1:]  # exclude self
    indices = indices[0][1:]
    
    # Hybrid score: combine distance (lower better) with popularity
    scores = []
    for idx, dist in zip(indices, distances):
        similarity = 1 / (1 + dist)  # convert distance to similarity
        hybrid_score = similarity * 0.7 + data.iloc[idx]['popularity_weight'] * 0.3
        scores.append((idx, hybrid_score))
    
    # Sort by hybrid score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    recommended_indices = [idx for idx, _ in scores[:n]]
    
    elapsed = time.time() - start_time
    print(f"Recommendation time: {elapsed:.4f}s")
    
    return recommended_indices

In [ ]:
# Function to create Spotify embed
def get_spotify_embed(uri):
    """Create Spotify embed iframe from track URI"""
    track_id = uri.split(':')[-1]
    embed_html = f'<iframe src="https://open.spotify.com/embed/track/{track_id}" width="100%" height="152" frameBorder="0" allowfullscreen="" allow="autoplay; clipboard-write; encrypted-media; fullscreen; picture-in-picture"></iframe>'
    return embed_html

# Create dropdown for track selection
track_options = [(f"{row['track_name']} by {row['track_artist']}", idx) for idx, row in data.iterrows()]
track_dropdown = widgets.Dropdown(
    options=track_options,
    description='Select Track:',
    layout=widgets.Layout(width='50%')
)

output = widgets.Output()

def on_track_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        track_idx = change['new']
        recs = get_recommendations(track_idx, 5)
        with output:
            output.clear_output()
            selected = data.iloc[track_idx]
            print(f"Selected: {selected['track_name']} by {selected['track_artist']} - Album: {selected['track_album_name']}, Genre: {selected['playlist_genre']} ({selected['playlist_subgenre']}), Year: {selected['year']}, Duration: {format_duration(selected['duration_ms'])}, Popularity: {selected['track_popularity']}")
            display(HTML(get_spotify_embed(selected['uri'])))
            print("\nRecommendations:")
            for i, rec_idx in enumerate(recs, 1):
                rec = data.iloc[rec_idx]
                print(f"{i}. {rec['track_name']} by {rec['track_artist']} - Album: {rec['track_album_name']}, Genre: {rec['playlist_genre']} ({rec['playlist_subgenre']}), Year: {rec['year']}, Duration: {format_duration(rec['duration_ms'])}, Popularity: {rec['track_popularity']}")
                display(HTML(get_spotify_embed(rec['uri'])))

track_dropdown.observe(on_track_change)

display(track_dropdown, output)

Dropdown(description='Select Track:', layout=Layout(width='50%'), options=(('Die With A Smile by Lady Gaga, Br…

Output()

: 

In [17]:
# Test recommendation time
test_idx = 0
recs = get_recommendations(test_idx, 5)
print("Test recommendations:")
for i, rec_idx in enumerate(recs, 1):
    rec = data.iloc[rec_idx]
    print(f"{i}. {rec['track_name']} by {rec['track_artist']} - Album: {rec['track_album_name']}, Genre: {rec['playlist_genre']} ({rec['playlist_subgenre']}), Year: {rec['year']}, Duration: {format_duration(rec['duration_ms'])}, Popularity: {rec['track_popularity']}")

Recommendation time: 0.0030s
Test recommendations:
1. Die With A Smile by Lady Gaga, Bruno Mars - Album: Die With A Smile, Genre: gaming (modern), Year: 2024.0, Duration: 4:11, Popularity: 100
2. Die With A Smile by Lady Gaga, Bruno Mars - Album: Die With A Smile, Genre: pop (mainstream), Year: 2024.0, Duration: 4:11, Popularity: 100
3. PRAYER (ADURA) by Biola Yakubu - Album: PRAYER (ADURA), Genre: gospel (modern), Year: 2024.0, Duration: 5:05, Popularity: 33
4. End of the Tunnel by A.Y. Bouzaglou - Album: End of the Tunnel, Genre: world (jewish), Year: 2019.0, Duration: 3:59, Popularity: 19
5. Alltid god by Impuls - Album: Drømmested, Genre: pop (scandi), Year: 2019.0, Duration: 4:41, Popularity: 13


## Conclusion

This notebook implements a hybrid music recommendation system using content-based filtering with Ball Tree for efficient nearest neighbor search. The system combines audio feature similarity with popularity weighting to provide personalized recommendations.

Key features:
- Fast recommendations (< 0.5s as required)
- Hybrid approach: 70% content similarity + 30% popularity
- Interactive widget for testing
- Scalable with Ball Tree for large datasets

For deployment as a web app, consider converting to Streamlit or Flask.